# RBF-SVM baseline: five seeds
JHub version of the original SVM baseline procedure.

It keeps the SVM methodology unchanged:

- same 5 outer folds;
- same train / validation / test roles;
- same 25 continuous + 3 categorical predictors;
- same feature masks and branch masks;
- same fixed 64-D fold-specific MRI embeddings from seed 17;
- same 32-configuration RBF-SVM grid;
- same 5-fold inner CV with search seed 17;
- same top-5 inner-CV candidates checked on the outer validation set;
- same final probability-enabled `SVC`;
- same 0.50 classification threshold.

The only experimental extension is:

```python
FINAL_SEEDS = [17, 42, 73, 101, 137]
```

Unlike the original notebook, this JHub version also saves **validation probabilities** for every fold/seed because they are required for leakage-aware fusion-weight selection later.

No Google Colab imports, Drive mounting, or `/content/...` paths are used.


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn

from IPython.display import display
from sklearn.metrics import (
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.preprocessing import OneHotEncoder
from sklearn.svm import SVC

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)

# JHub paths confirmed by the successful validation/fusion notebook

DATA_ROOT = Path("/home/jovyan/thesis_data/adni_mri/adni_mri")

INPUT_ROOT = (
    DATA_ROOT
    / "models"
    / "final_task_ready_inputs"
    / "mci_prognosis"
)

MRI_EMBEDDING_ROOT = (
    Path("/home/jovyan/thesis_runs")
    / "svm_rbf_predictions"
    / "mri_embeddings"
)

OUTPUT_ROOT = (
    Path("/home/jovyan/thesis_runs")
    / "svm_rbf_predictions_5seeds"
)

SEARCH_RESULTS_DIR = OUTPUT_ROOT / "search_results"
PREDICTIONS_DIR = OUTPUT_ROOT / "fold_predictions"
TABLES_DIR = OUTPUT_ROOT / "tables"
METADATA_DIR = OUTPUT_ROOT / "metadata"

for d in [
    OUTPUT_ROOT,
    SEARCH_RESULTS_DIR,
    PREDICTIONS_DIR,
    TABLES_DIR,
    METADATA_DIR,
]:
    d.mkdir(parents=True, exist_ok=True)

# Experiment contract

RID_COL = "RID"
TARGET_COL = "MCI_PROGNOSIS_TARGET"
OUTER_FOLD_COL = "OUTER_FOLD"
ROLE_COL = "DATA_ROLE"

OUTER_FOLDS = [0, 1, 2, 3, 4]
FINAL_SEEDS = [17, 42, 73, 101, 137]

SEARCH_SEED = 17
THRESHOLD = 0.50

INNER_CV_SPLITS = 5
TOP_K_VALIDATION = 5
N_JOBS = -1

RESUME_EXISTING_RUN = True

MRI_EMBEDDING_SEED = 17
MRI_EMBEDDING_DIM = 64
MRI_EMBEDDING_COLS = [
    f"MRI_EMB_{i:02d}"
    for i in range(MRI_EMBEDDING_DIM)
]

print("Python:", __import__("sys").version.split()[0])
print("scikit-learn:", sklearn.__version__)
print("Input root:", INPUT_ROOT)
print("MRI embedding root:", MRI_EMBEDDING_ROOT)
print("Output root:", OUTPUT_ROOT)
print("Final seeds:", FINAL_SEEDS)


## 1. Exact predictor contract


In [ ]:
CONTINUOUS_COLS = [
    "DEMOGRAPHICS__AGE_AT_BASELINE__Z",
    "DEMOGRAPHICS__PTEDUCAT_CLEAN__Z",
    "ADAS__TOTSCORE__Z",
    "ADAS__TOTAL13__Z",
    "MMSE__MMSE_TOTAL_SCORE__Z",
    "MMSE__MMSE_ORIENTATION_SCORE__Z",
    "MMSE__MMSE_REGISTRATION_SCORE__Z",
    "MMSE__MMSE_ATTENTION_SCORE__Z",
    "MMSE__MMSE_DELAYED_RECALL_SCORE__Z",
    "MMSE__MMSE_LANGUAGE_COMMAND_SCORE__Z",
    "FAQ__FAQTOTAL__Z",
    "CSF__ABETA40__Z",
    "CSF__ABETA42__Z",
    "CSF__TAU__Z",
    "CSF__PTAU__Z",
    "CSF__ABETA42_40_RATIO__Z",
    "PLASMA__pT217_F__Z",
    "PLASMA__AB42_F__Z",
    "PLASMA__AB40_F__Z",
    "PLASMA__AB42_AB40_F__Z",
    "PLASMA__pT217_AB42_F__Z",
    "PLASMA__NfL_Q__Z",
    "PLASMA__GFAP_Q__Z",
    "PLASMA__NfL_F__Z",
    "PLASMA__GFAP_F__Z",
]

CATEGORICAL_COLS = [
    "DEMOGRAPHICS__PTGENDER_CLEAN__IDX",
    "DEMOGRAPHICS__PTHAND_CLEAN__IDX",
    "APOE__APOE4_ALLELE_COUNT__IDX",
]

BRANCH_MASK_COLS = [
    "BRANCH_MASK__DEMOGRAPHICS",
    "BRANCH_MASK__COGNITIVE_FUNCTIONAL",
    "BRANCH_MASK__CSF",
    "BRANCH_MASK__PLASMA",
    "BRANCH_MASK__APOE",
]

MRI_BRANCH_MASK_COL = "BRANCH_MASK__MRI"

TABULAR_PREDICTOR_COLS = CONTINUOUS_COLS + CATEGORICAL_COLS

def predictor_to_feature_mask(predictor: str) -> str:
    base = predictor
    for suffix in ("__Z", "__IDX"):
        if base.endswith(suffix):
            base = base[:-len(suffix)]
            break
    return "FEATURE_MASK__" + base.replace("__", "_")

EXPECTED_FEATURE_MASKS = [
    predictor_to_feature_mask(c)
    for c in TABULAR_PREDICTOR_COLS
]

assert len(CONTINUOUS_COLS) == 25
assert len(CATEGORICAL_COLS) == 3
assert len(EXPECTED_FEATURE_MASKS) == 28

print("Continuous:", len(CONTINUOUS_COLS))
print("Categorical:", len(CATEGORICAL_COLS))
print("Feature masks:", len(EXPECTED_FEATURE_MASKS))


## 2. Input checks

The paths below are the ones that were actually discovered in the successful JHub validation/fusion run:

- fold tables: `/home/jovyan/thesis_data/adni_mri/adni_mri/models/final_task_ready_inputs/mci_prognosis`
- MRI embeddings: `/home/jovyan/thesis_runs/svm_rbf_predictions/mri_embeddings`


In [ ]:
def fold_path(fold: int) -> Path:
    candidates = [
        INPUT_ROOT
        / f"mci_prognosis_outer_fold_{fold}_final_task_ready.csv",

        INPUT_ROOT
        / f"mci_prognosis_outer_fold_{fold}_categorical_encoded.csv",
    ]

    for p in candidates:
        if p.exists():
            return p

    raise FileNotFoundError(
        f"No prepared fold file found for fold {fold} under {INPUT_ROOT}"
    )

def mri_embedding_path(fold: int) -> Path:
    return (
        MRI_EMBEDDING_ROOT
        / f"mci_prognosis_fold_{fold}_seed_{MRI_EMBEDDING_SEED}_mri_embeddings.csv"
    )

missing = []

for fold in OUTER_FOLDS:
    try:
        fp = fold_path(fold)
    except FileNotFoundError as e:
        missing.append(str(e))

    mp = mri_embedding_path(fold)
    if not mp.exists():
        missing.append(f"Missing MRI embedding: {mp}")

if missing:
    print("PRE-FLIGHT FAILED:")
    for x in missing:
        print(" -", x)
    raise FileNotFoundError(
        f"{len(missing)} required input(s) are missing."
    )

print("Input check passed.")
for fold in OUTER_FOLDS:
    print(
        f"fold {fold}:",
        fold_path(fold).name,
        "|",
        mri_embedding_path(fold).name,
    )


## 3. Load and validate the five outer folds


In [ ]:
EXPECTED_ROLE_COUNTS = {
    0: {"train": 369, "validation": 66, "test": 109},
    1: {"train": 369, "validation": 66, "test": 109},
    2: {"train": 369, "validation": 66, "test": 109},
    3: {"train": 369, "validation": 66, "test": 109},
    4: {"train": 370, "validation": 66, "test": 108},
}

def _coerce_binary_mask(s: pd.Series, name: str) -> pd.Series:
    out = pd.to_numeric(
        s,
        errors="raise",
    ).astype(int)

    bad = set(out.unique()) - {0, 1}

    if bad:
        raise ValueError(
            f"{name} contains values outside 0/1: {sorted(bad)}"
        )

    return out

def load_and_validate_fold(fold: int) -> pd.DataFrame:
    path = fold_path(fold)
    df = pd.read_csv(path)

    # Newer JHub final-task files use MODEL_TARGET.
    if TARGET_COL not in df.columns:
        if "MODEL_TARGET" in df.columns:
            df[TARGET_COL] = pd.to_numeric(
                df["MODEL_TARGET"],
                errors="raise",
            ).astype(int)
        else:
            raise KeyError(
                f"{path}: neither {TARGET_COL} nor MODEL_TARGET exists."
            )

    if OUTER_FOLD_COL not in df.columns:
        df[OUTER_FOLD_COL] = fold

    required = {
        RID_COL,
        TARGET_COL,
        OUTER_FOLD_COL,
        ROLE_COL,
        *TABULAR_PREDICTOR_COLS,
        *EXPECTED_FEATURE_MASKS,
        *BRANCH_MASK_COLS,
        MRI_BRANCH_MASK_COL,
    }

    missing = sorted(
        required - set(df.columns)
    )

    if missing:
        raise ValueError(
            f"Fold {fold}: required columns missing:\n{missing}"
        )

    if len(df) != 544:
        raise ValueError(
            f"Fold {fold}: expected 544 rows, found {len(df)}"
        )

    if df[RID_COL].duplicated().any():
        raise ValueError(
            f"Fold {fold}: duplicate RID values found."
        )

    df[RID_COL] = pd.to_numeric(
        df[RID_COL],
        errors="raise",
    ).astype(int)

    df[TARGET_COL] = pd.to_numeric(
        df[TARGET_COL],
        errors="raise",
    ).astype(int)

    if set(df[TARGET_COL].unique()) != {0, 1}:
        raise ValueError(
            f"Fold {fold}: expected binary targets {{0,1}}."
        )

    df[ROLE_COL] = (
        df[ROLE_COL]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    actual_counts = (
        df[ROLE_COL]
        .value_counts()
        .to_dict()
    )

    for role, expected_n in EXPECTED_ROLE_COUNTS[fold].items():
        actual_n = int(
            actual_counts.get(role, 0)
        )

        if actual_n != expected_n:
            raise ValueError(
                f"Fold {fold}: {role} expected {expected_n}, found {actual_n}"
            )

    for c in (
        EXPECTED_FEATURE_MASKS
        + BRANCH_MASK_COLS
        + [MRI_BRANCH_MASK_COL]
    ):
        df[c] = _coerce_binary_mask(
            df[c],
            c,
        )

    for predictor in CONTINUOUS_COLS:
        mask_col = predictor_to_feature_mask(
            predictor
        )

        numeric = pd.to_numeric(
            df[predictor],
            errors="coerce",
        )

        observed_but_nan = (
            (df[mask_col] == 1)
            & numeric.isna()
        )

        if observed_but_nan.any():
            rids = (
                df.loc[
                    observed_but_nan,
                    RID_COL,
                ]
                .head()
                .tolist()
            )

            raise ValueError(
                f"Fold {fold}: {predictor} is observed by mask "
                f"but NaN for RIDs {rids}"
            )

        numeric = numeric.fillna(0.0)
        numeric.loc[
            df[mask_col] == 0
        ] = 0.0

        df[predictor] = numeric

    for predictor in CATEGORICAL_COLS:
        mask_col = predictor_to_feature_mask(
            predictor
        )

        df[predictor] = pd.to_numeric(
            df[predictor],
            errors="raise",
        ).astype(int)

        df.loc[
            df[mask_col] == 0,
            predictor,
        ] = 0

    return df

fold_tables = {
    fold: load_and_validate_fold(fold)
    for fold in OUTER_FOLDS
}

summary_rows = []

for fold, df in fold_tables.items():
    counts = df[ROLE_COL].value_counts()

    summary_rows.append({
        "fold": fold,
        "n_total": len(df),
        "n_train": int(counts["train"]),
        "n_validation": int(counts["validation"]),
        "n_test": int(counts["test"]),
        "n_sMCI": int((df[TARGET_COL] == 0).sum()),
        "n_pMCI": int((df[TARGET_COL] == 1).sum()),
    })

display(pd.DataFrame(summary_rows))


## 4. Join the fixed 64-D MRI embeddings and construct matrices


In [ ]:
def load_and_validate_mri_embeddings(
    fold: int,
    fold_df: pd.DataFrame,
) -> pd.DataFrame:

    path = mri_embedding_path(fold)
    emb = pd.read_csv(path)

    required = {
        RID_COL,
        MRI_BRANCH_MASK_COL,
        *MRI_EMBEDDING_COLS,
    }

    missing = sorted(
        required - set(emb.columns)
    )

    if missing:
        raise ValueError(
            f"Fold {fold}: MRI embedding file missing columns:\n{missing}"
        )

    if len(emb) != 544:
        raise ValueError(
            f"Fold {fold}: expected 544 embedding rows, found {len(emb)}"
        )

    emb[RID_COL] = pd.to_numeric(
        emb[RID_COL],
        errors="raise",
    ).astype(int)

    if emb[RID_COL].duplicated().any():
        raise ValueError(
            f"Fold {fold}: duplicate RID values in MRI embeddings."
        )

    emb[MRI_BRANCH_MASK_COL] = _coerce_binary_mask(
        emb[MRI_BRANCH_MASK_COL],
        MRI_BRANCH_MASK_COL,
    )

    for col in MRI_EMBEDDING_COLS:
        emb[col] = pd.to_numeric(
            emb[col],
            errors="raise",
        ).astype(float)

    values = emb[
        MRI_EMBEDDING_COLS
    ].to_numpy(dtype=np.float64)

    if not np.isfinite(values).all():
        raise ValueError(
            f"Fold {fold}: non-finite MRI embedding values."
        )

    if set(emb[RID_COL]) != set(fold_df[RID_COL]):
        raise ValueError(
            f"Fold {fold}: RID mismatch between fold table and MRI embeddings."
        )

    return emb[
        [RID_COL, *MRI_EMBEDDING_COLS]
    ].copy()

def make_outer_fold_matrices(
    fold: int,
) -> dict:

    df = fold_tables[
        fold
    ].copy()

    emb = load_and_validate_mri_embeddings(
        fold,
        df,
    )

    df = df.merge(
        emb,
        on=RID_COL,
        how="left",
        validate="one_to_one",
    )

    train = df.loc[
        df[ROLE_COL] == "train"
    ].copy()

    val = df.loc[
        df[ROLE_COL] == "validation"
    ].copy()

    test = df.loc[
        df[ROLE_COL] == "test"
    ].copy()

    try:
        ohe = OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False,
            dtype=np.float64,
        )
    except TypeError:
        ohe = OneHotEncoder(
            handle_unknown="ignore",
            sparse=False,
            dtype=np.float64,
        )

    ohe.fit(
        train[CATEGORICAL_COLS]
    )

    def transform(part: pd.DataFrame):
        x_cont = part[
            CONTINUOUS_COLS
        ].to_numpy(dtype=np.float64)

        x_cat = ohe.transform(
            part[CATEGORICAL_COLS]
        )

        x_mri = part[
            MRI_EMBEDDING_COLS
        ].to_numpy(dtype=np.float64)

        extra_cols = (
            EXPECTED_FEATURE_MASKS
            + BRANCH_MASK_COLS
            + [MRI_BRANCH_MASK_COL]
        )

        x_extra = part[
            extra_cols
        ].to_numpy(dtype=np.float64)

        X = np.concatenate(
            [
                x_cont,
                x_cat,
                x_mri,
                x_extra,
            ],
            axis=1,
        )

        y = part[
            TARGET_COL
        ].to_numpy(dtype=int)

        rid = part[
            RID_COL
        ].to_numpy(dtype=int)

        return X, y, rid

    X_train, y_train, rid_train = transform(train)
    X_val, y_val, rid_val = transform(val)
    X_test, y_test, rid_test = transform(test)

    return {
        "X_train": X_train,
        "y_train": y_train,
        "rid_train": rid_train,
        "X_val": X_val,
        "y_val": y_val,
        "rid_val": rid_val,
        "X_test": X_test,
        "y_test": y_test,
        "rid_test": rid_test,
    }

example = make_outer_fold_matrices(0)

print("Fold 0 train:", example["X_train"].shape)
print("Fold 0 validation:", example["X_val"].shape)
print("Fold 0 test:", example["X_test"].shape)
print("Expected transformed feature count: 133")


## 5. Exact RBF-SVM search

Same search as the original notebook:

- `C = [0.1, 1, 10, 100]`
- `gamma = ["scale", 0.001, 0.01, 0.1]`
- `class_weight = [None, "balanced"]`
- 32 candidates × 5 inner folds = 160 fits per outer fold
- rank by inner-CV ROC AUC, then AP
- take the top 5
- fit each top candidate on the outer training set
- choose among them by outer-validation ROC AUC, then AP, then inner-CV ROC AUC


In [ ]:
SVM_C_VALUES = [
    0.1,
    1.0,
    10.0,
    100.0,
]

SVM_GAMMA_VALUES = [
    "scale",
    0.001,
    0.01,
    0.1,
]

INNER_CV = StratifiedKFold(
    n_splits=INNER_CV_SPLITS,
    shuffle=True,
    random_state=SEARCH_SEED,
)

SCORING = {
    "roc_auc": "roc_auc",
    "ap": "average_precision",
}

def json_safe(obj):
    if isinstance(obj, (np.integer,)):
        return int(obj)

    if isinstance(obj, (np.floating,)):
        return float(obj)

    if isinstance(obj, np.ndarray):
        return obj.tolist()

    if isinstance(obj, dict):
        return {
            str(k): json_safe(v)
            for k, v in obj.items()
        }

    if isinstance(obj, (list, tuple)):
        return [
            json_safe(v)
            for v in obj
        ]

    return obj

def top_candidates(
    cv_results,
    top_k=TOP_K_VALIDATION,
):
    df = pd.DataFrame(
        cv_results
    ).copy()

    return (
        df.sort_values(
            [
                "mean_test_roc_auc",
                "mean_test_ap",
            ],
            ascending=[
                False,
                False,
            ],
        )
        .head(top_k)
        .reset_index(drop=True)
    )

def validation_select(
    top_df,
    X_train,
    y_train,
    X_val,
    y_val,
):
    rows = []

    for rank, row in top_df.iterrows():
        params = dict(
            row["params"]
        )

        model = SVC(
            kernel="rbf",
            probability=False,
            cache_size=2000,
            **params,
        )

        model.fit(
            X_train,
            y_train,
        )

        scores = model.decision_function(
            X_val
        )

        rows.append({
            "inner_cv_rank":
                rank + 1,

            "inner_cv_roc_auc":
                float(row["mean_test_roc_auc"]),

            "inner_cv_ap":
                float(row["mean_test_ap"]),

            "validation_roc_auc":
                float(
                    roc_auc_score(
                        y_val,
                        scores,
                    )
                ),

            "validation_ap":
                float(
                    average_precision_score(
                        y_val,
                        scores,
                    )
                ),

            "params":
                params,
        })

    ranking = (
        pd.DataFrame(rows)
        .sort_values(
            [
                "validation_roc_auc",
                "validation_ap",
                "inner_cv_roc_auc",
            ],
            ascending=[
                False,
                False,
                False,
            ],
        )
        .reset_index(drop=True)
    )

    return (
        dict(
            ranking.iloc[0]["params"]
        ),
        ranking,
    )

def tune_svm(
    fold,
    X_train,
    y_train,
    X_val,
    y_val,
):
    selected_json = (
        SEARCH_RESULTS_DIR
        / f"fold_{fold}_svm_selected.json"
    )

    if (
        RESUME_EXISTING_RUN
        and selected_json.exists()
    ):
        print(
            f"[resume] selected SVM exists for fold {fold}"
        )

        with open(
            selected_json,
            "r",
        ) as f:
            return json.load(f)

    base = SVC(
        kernel="rbf",
        probability=False,
        cache_size=2000,
    )

    grid = {
        "C": SVM_C_VALUES,
        "gamma": SVM_GAMMA_VALUES,
        "class_weight": [
            None,
            "balanced",
        ],
    }

    search = GridSearchCV(
        base,
        grid,
        scoring=SCORING,
        refit=False,
        cv=INNER_CV,
        n_jobs=N_JOBS,
        error_score="raise",
    )

    search.fit(
        X_train,
        y_train,
    )

    cv_df = pd.DataFrame(
        search.cv_results_
    ).copy()

    if "params" in cv_df.columns:
        cv_df["params_json"] = (
            cv_df["params"]
            .apply(
                lambda p: json.dumps(
                    json_safe(p),
                    sort_keys=True,
                )
            )
        )
        cv_df = cv_df.drop(
            columns=["params"]
        )

    cv_df.to_csv(
        SEARCH_RESULTS_DIR
        / f"fold_{fold}_svm_inner_cv.csv",
        index=False,
    )

    top = top_candidates(
        search.cv_results_
    )

    best_params, ranking = validation_select(
        top,
        X_train,
        y_train,
        X_val,
        y_val,
    )

    ranking_out = ranking.copy()
    ranking_out["params_json"] = (
        ranking_out["params"]
        .apply(
            lambda p: json.dumps(
                json_safe(p),
                sort_keys=True,
            )
        )
    )

    ranking_out.drop(
        columns=["params"]
    ).to_csv(
        SEARCH_RESULTS_DIR
        / f"fold_{fold}_svm_validation_ranking.csv",
        index=False,
    )

    payload = {
        "model": "svm_rbf",
        "fold": int(fold),
        "selected_params":
            json_safe(best_params),
        "search_seed":
            SEARCH_SEED,
    }

    with open(
        selected_json,
        "w",
    ) as f:
        json.dump(
            payload,
            f,
            indent=2,
        )

    return payload


## 6. Five-seed final fits

The final SVM is fitted on the **outer training split only**, exactly like the original baseline.

For each seed, this cell saves:

- outer-validation probabilities;
- held-out outer-test probabilities.

The test CSV naming remains compatible with the old SVM files.


In [ ]:
def save_prediction_file(
    path,
    fold,
    seed,
    rids,
    y_true,
    y_prob,
):
    out = pd.DataFrame({
        "MODEL":
            "svm_rbf",

        "SEED":
            int(seed),

        "FOLD":
            int(fold),

        "RID":
            np.asarray(
                rids,
                dtype=int,
            ),

        "TARGET":
            np.asarray(
                y_true,
                dtype=int,
            ),

        "PROB_PMCI":
            np.asarray(
                y_prob,
                dtype=float,
            ),
    })

    out[
        "PREDICTED_CLASS_0P5"
    ] = (
        out["PROB_PMCI"]
        >= THRESHOLD
    ).astype(int)

    if out["RID"].duplicated().any():
        raise ValueError(
            f"Duplicate RIDs in {path.name}"
        )

    out.to_csv(
        path,
        index=False,
    )

selected_rows = []

for fold in OUTER_FOLDS:
    print("\n" + "=" * 88)
    print("OUTER FOLD", fold)
    print("=" * 88)

    M = make_outer_fold_matrices(
        fold
    )

    payload = tune_svm(
        fold,
        M["X_train"],
        M["y_train"],
        M["X_val"],
        M["y_val"],
    )

    params = dict(
        payload["selected_params"]
    )

    selected_rows.append({
        "FOLD":
            fold,

        "SELECTED_PARAMS":
            json.dumps(
                params,
                sort_keys=True,
            ),
    })

    print(
        "Selected params:",
        params,
    )

    for seed in FINAL_SEEDS:
        test_path = (
            PREDICTIONS_DIR
            / f"svm_rbf_fold_{fold}_seed_{seed}_test_predictions.csv"
        )

        val_path = (
            PREDICTIONS_DIR
            / f"svm_rbf_fold_{fold}_seed_{seed}_validation_predictions.csv"
        )

        if (
            RESUME_EXISTING_RUN
            and test_path.exists()
            and val_path.exists()
        ):
            print(
                f"[resume] fold={fold} seed={seed}"
            )
            continue

        print(
            f"Final fit: fold={fold} seed={seed}"
        )

        model = SVC(
            kernel="rbf",
            probability=True,
            cache_size=2000,
            random_state=seed,
            **params,
        )

        model.fit(
            M["X_train"],
            M["y_train"],
        )

        val_prob = model.predict_proba(
            M["X_val"]
        )[:, 1]

        test_prob = model.predict_proba(
            M["X_test"]
        )[:, 1]

        save_prediction_file(
            val_path,
            fold,
            seed,
            M["rid_val"],
            M["y_val"],
            val_prob,
        )

        save_prediction_file(
            test_path,
            fold,
            seed,
            M["rid_test"],
            M["y_test"],
            test_prob,
        )

selected_df = pd.DataFrame(
    selected_rows
)

display(selected_df)

selected_df.to_csv(
    TABLES_DIR
    / "svm_selected_params_by_fold.csv",
    index=False,
)


## 7. Aggregate the five held-out test folds for each seed


In [ ]:
def metrics_from_prob(
    y_true,
    y_prob,
):
    y_true = np.asarray(
        y_true,
        dtype=int,
    )

    y_prob = np.asarray(
        y_prob,
        dtype=float,
    )

    pred = (
        y_prob
        >= THRESHOLD
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        pred,
        labels=[0, 1],
    ).ravel()

    sensitivity = (
        tp / (tp + fn)
        if (tp + fn)
        else np.nan
    )

    specificity = (
        tn / (tn + fp)
        if (tn + fp)
        else np.nan
    )

    return {
        "roc_auc":
            float(
                roc_auc_score(
                    y_true,
                    y_prob,
                )
            ),

        "average_precision":
            float(
                average_precision_score(
                    y_true,
                    y_prob,
                )
            ),

        "balanced_accuracy":
            float(
                balanced_accuracy_score(
                    y_true,
                    pred,
                )
            ),

        "sensitivity":
            float(sensitivity),

        "specificity":
            float(specificity),

        "f1":
            float(
                f1_score(
                    y_true,
                    pred,
                )
            ),

        "brier":
            float(
                brier_score_loss(
                    y_true,
                    y_prob,
                )
            ),

        "n":
            int(len(y_true)),
    }

seed_metric_rows = []
pooled_frames = []

for seed in FINAL_SEEDS:
    parts = []

    for fold in OUTER_FOLDS:
        path = (
            PREDICTIONS_DIR
            / f"svm_rbf_fold_{fold}_seed_{seed}_test_predictions.csv"
        )

        if not path.exists():
            raise FileNotFoundError(
                path
            )

        parts.append(
            pd.read_csv(path)
        )

    pooled = pd.concat(
        parts,
        ignore_index=True,
    )

    if (
        len(pooled) != 544
        or pooled["RID"].nunique() != 544
    ):
        raise ValueError(
            f"Seed {seed}: expected exactly 544 unique OOF participants."
        )

    pooled_path = (
        PREDICTIONS_DIR
        / f"svm_rbf_seed_{seed}_pooled_out_of_fold_predictions.csv"
    )

    pooled.to_csv(
        pooled_path,
        index=False,
    )

    pooled_frames.append(
        pooled
    )

    m = metrics_from_prob(
        pooled["TARGET"],
        pooled["PROB_PMCI"],
    )

    seed_metric_rows.append({
        "model":
            "svm_rbf",

        "seed":
            seed,

        **m,
    })

seed_metrics = pd.DataFrame(
    seed_metric_rows
)

display(seed_metrics)

seed_metrics.to_csv(
    TABLES_DIR
    / "svm_rbf_pooled_metrics_by_seed.csv",
    index=False,
)

all_oof = pd.concat(
    pooled_frames,
    ignore_index=True,
)

all_oof.to_csv(
    PREDICTIONS_DIR
    / "all_svm_rbf_5seed_out_of_fold_predictions.csv",
    index=False,
)

summary = pd.DataFrame([{
    "model":
        "svm_rbf",

    "n_seeds":
        len(FINAL_SEEDS),

    "roc_auc_mean":
        seed_metrics["roc_auc"].mean(),

    "roc_auc_sd":
        seed_metrics["roc_auc"].std(ddof=1),

    "average_precision_mean":
        seed_metrics["average_precision"].mean(),

    "average_precision_sd":
        seed_metrics["average_precision"].std(ddof=1),

    "balanced_accuracy_mean":
        seed_metrics["balanced_accuracy"].mean(),

    "balanced_accuracy_sd":
        seed_metrics["balanced_accuracy"].std(ddof=1),

    "brier_mean":
        seed_metrics["brier"].mean(),

    "brier_sd":
        seed_metrics["brier"].std(ddof=1),
}])

display(summary)

summary.to_csv(
    TABLES_DIR
    / "svm_rbf_5seed_mean_sd.csv",
    index=False,
)


## 8. Five-seed probability ensemble across the 544 OOF participants

This is **additional context**, not a replacement for the per-seed baseline summary.

For each participant, the five seed probabilities are averaged, then one pooled ROC AUC is calculated across all 544 held-out participants.


In [ ]:
ensemble = (
    all_oof
    .groupby("RID")
    .agg(
        TARGET=("TARGET", "first"),
        PROB_PMCI=("PROB_PMCI", "mean"),
        PROB_PMCI_SD=("PROB_PMCI", "std"),
    )
    .reset_index()
)

assert len(ensemble) == 544
assert ensemble["RID"].nunique() == 544

ensemble_metrics = metrics_from_prob(
    ensemble["TARGET"],
    ensemble["PROB_PMCI"],
)

print("Five-seed SVM probability ensemble:")
for k, v in ensemble_metrics.items():
    print(f"{k}: {v}")

ensemble.to_csv(
    PREDICTIONS_DIR
    / "svm_rbf_5seed_probability_ensemble_oof.csv",
    index=False,
)

pd.DataFrame(
    [ensemble_metrics]
).to_csv(
    TABLES_DIR
    / "svm_rbf_5seed_probability_ensemble_metrics.csv",
    index=False,
)


## 9. Output paths for the next five-seed fusion notebook

The important files will be under:

```text
/home/jovyan/thesis_runs/svm_rbf_predictions_5seeds/fold_predictions/
```

There should be:

- 25 test files = 5 folds × 5 seeds;
- 25 validation files = 5 folds × 5 seeds;
- 5 pooled OOF seed files;
- one five-seed SVM probability ensemble file.

These validation files let us rerun the SVM+TMC and SVM+Hybrid fusion **properly over all five matched seeds**.


In [ ]:
print("Finished.")
print("Predictions:", PREDICTIONS_DIR)
print("Tables:", TABLES_DIR)

test_files = sorted(
    PREDICTIONS_DIR.glob(
        "svm_rbf_fold_*_seed_*_test_predictions.csv"
    )
)

validation_files = sorted(
    PREDICTIONS_DIR.glob(
        "svm_rbf_fold_*_seed_*_validation_predictions.csv"
    )
)

print("Test fold/seed files:", len(test_files))
print("Validation fold/seed files:", len(validation_files))

assert len(test_files) == 25
assert len(validation_files) == 25
